In [ ]:
import pandas as pd
import numpy as np

In [22]:
#### get sub-folder directories

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Base URL (this is like your "parent directory")
parent_url = "https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/"

#adding .text folder
text_folder = "Text_Files"

# Download the webpage that lists the files/folders at the URL
html = requests.get(parent_url).text

# Parse the HTML so we can extract links (<a href="...">)
soup = BeautifulSoup(html, "html.parser")

# Store directory links here
subdirs = []

# Loop through every hyperlink on the page
for link in soup.find_all("a"):
    href = link.get("href")           # extract the URL inside href="..."

    # If it ends in "/", it is usually a directory in web listings
    if href and href.endswith("/"):
        full_url = urljoin(parent_url, href)   # convert relative path into full URL
        full_url_text = urljoin(full_url, text_folder)
        subdirs.append(full_url_text)
        
# Print all found subdirectories
print(subdirs)

['https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/CAPRA/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/CAUBC/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/CAWTA/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/CYYD/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/CYZY/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USALY/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USCLL/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USCOU/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USIUB/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USOHS/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USOSU/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USRAL/Text_Files', 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/

In [ ]:
['https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/CAPRA/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/CAUBC/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/CAWTA/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/CYYD/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/CYZY/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USALY/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USCLL/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USCOU/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USIUB/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USOHS/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USOSU/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USRAL/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USSOM/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USUND/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USVAL/Text_Files', 
 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USVPI/Text_Files']

In [35]:
#### have to do it one by one - This One works!!
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from io import StringIO

path = 'https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USVAL/Text_Files/'


html = requests.get(path).text
soup = BeautifulSoup(html, "html.parser")

txt_files = []
for link in soup.find_all("a"):
    href = link.get("href")
    if href and href.endswith(".txt"):
        txt_files.append(urljoin(path, href))

df_list = []

for file_url in txt_files:
    r = requests.get(file_url)
    file_text = r.text.strip()

    # skip empty files
    if len(file_text) == 0:
        print("Skipping empty file:", file_url)
        continue

    # read whitespace-delimited text
    df = pd.read_csv(StringIO(file_text),
                     sep=",", 
                     header=None, 
                     engine="python")   # no header row assumed

    df["source_file"] = file_url  # optional: track where it came from
    df_list.append(df)

# --- combine everything ---
combined_df = pd.concat(df_list, ignore_index=True)
combined_df.columns = ['Height(m)', 'Pressure(hPa)', 'Temperature(C)', 'DewPoint(C)', 'WindDirection(degrees)', 'WindSpeed(m/s)', 'Source']

# --- set first column as index ---
combined_df.set_index('Height(m)', inplace=True)

# parsing data by year, date, hour and minute

# inconsistency in file names
pattern = r'(?P<Year>\d{4})_?(?P<Month>\d{2})(?P<Day>\d{2})(?:_(?P<Hour>\d{2})(?P<Minute>\d{2}))?'

extracted = combined_df['Source'].str.extract(pattern)

combined_df = pd.concat([combined_df, extracted], axis=1)

combined_df['Datetime'] = pd.to_datetime(
    combined_df[['Year', 'Month', 'Day', 'Hour', 'Minute']],
    errors='coerce'
)

# calculate water vapor pressure (hpa)
e_0 = 6.113 #hpa
Lv_Rv = 5423 # k, for liquid water L_v/R_v actually L_v is latent heat of evaporation, R_v is water vapor gas constant
t_0 = 273.15 # k
combined_df['WaterVaporPressure(hpa)'] = e_0 * np.exp(Lv_Rv*(1/t_0 - 1/(combined_df['DewPoint(C)'] + 273.15)))

# calculate water vapor density (kg/m^3)
R_v = 461.525 # K^-1*kg^-1 water vapor gas constant
combined_df['WaterVaporDensity(kg/m^3)'] = (combined_df['WaterVaporPressure(hpa)']*100)/(R_v*(combined_df['Temperature(C)'] + 273.15))



# output filename based on folder name
folder_name = path.strip("/").split("/")[-2]


combined_df['Location'] = np.array(folder_name)


output_file = r"C:\Users\lisaw\OneDrive\Documents\temp\ThesisRepo\Data\radiosonde\\" + f"{folder_name}_radiosonde.csv.gz"

# save compressed
combined_df.to_csv(output_file, index=True, compression="gzip")

print(f"Saved: {output_file}")

print(combined_df)

Skipping empty file: https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USVAL/Text_Files/USVAL_20260123_2100.txt
Skipping empty file: https://cw3e-datashare.ucsd.edu/CW3E_Radiosondes/UCRP/USVAL/Text_Files/USVAL_20260124_0000.txt
Saved: C:\Users\lisaw\OneDrive\Documents\temp\ThesisRepo\Data\radiosonde\\USVAL_radiosonde.csv.gz
           Pressure(hPa)  Temperature(C)  DewPoint(C)  WindDirection(degrees)  \
Height(m)                                                                       
258.00            979.59            8.70         3.05                    0.00   
295.71            975.09            4.83         2.63                  327.98   
304.27            974.07            4.67         2.59                  330.40   
312.66            973.07            4.57         2.59                  329.37   
319.52            972.25            4.57         2.59                  329.55   
...                  ...             ...          ...                     ...   
16479.41           96.